Цель эксперимента: Обучить и сохранить альтернативные продвинутые модели ансамблирования — алгоритмы CatBoost и XGBoost, с учетом их специфических требований к подготовке категориальных данных.
Какие данные используются: Датасет cleaned_bnpl_data.csv (в XGBoost подавались dummy-переменные, в CatBoost — данные со строгим приведением типов .astype('category')).
Какие основные выводы: Обе модели успешно обучены и сохранены в артефакты (catboost_model.pkl и xgboost_model.pkl). CatBoost продемонстрировал удобство нативной работы с категориями «из коробки», а XGBoost показал высокую скорость сходимости на разреженной матрице флагов.

In [3]:
import pandas as pd
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import joblib
import os

from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
print('Working dir:', Path.cwd())

df = pd.read_csv('data/cleaned_bnpl_data.csv')
if 'Target' not in df.columns:
    if 'Repayment_Status' in df.columns:
        df['Target'] = (df['Repayment_Status'] == 'Defaulted').astype(int)
    else:
        raise KeyError('Target not found in data/cleaned_bnpl_data.csv')
X = df.drop(columns=['Target'])

y = df['Target']

cat_cols = ['Gender', 'Purchase_Category', 'BNPL_Provider', 'Device_Type', 'Connection_Type', 'Browser']
X_cat = X.copy()
for col in cat_cols:
    X_cat[col] = X_cat[col].astype('category')

X_xgb = pd.get_dummies(X, columns=cat_cols)

os.makedirs('artifacts/models', exist_ok=True)

cat_model = CatBoostClassifier(iterations=100, learning_rate=0.1, cat_features=cat_cols, verbose=False)
cat_model.fit(X_cat, y)

xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6)
xgb_model.fit(X_xgb, y)
joblib.dump(cat_model, 'artifacts/models/catboost_model.pkl')

xgb_model.fit(X_xgb, y)
joblib.dump(xgb_model, 'artifacts/models/xgboost_model.pkl')


Working dir: c:\Users\fedor\Documents\proga\pet_scoring


['artifacts/models/xgboost_model.pkl']